# Data Preprocessing & Feature Engineering

This notebook transforms raw half‑hourly waste records into **daily** datasets ready for forecasting with:
- XGBoost / LightGBM
- SARIMA / SARIMAX
- Prophet
- Simple Moving Average (baseline)

All models work on a **daily** granularity – sub‑daily observations are too noisy for a 7‑day‑ahead forecast. We aggregate per `(date, canteen section)` and engineer calendar, lag, and rolling features.

In [194]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Create output directory for per-section CSVs
os.makedirs('data/per_section', exist_ok=True)

## Step 1 — Load Raw Data & Parse Timestamps

In [195]:
df = pd.read_csv("data/synthetic_forecast_cleaned.csv")
print(f"Raw shape: {df.shape}")

# Parse timestamp and set as index for convenience
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

# Sort by time and location
df.sort_index(inplace=True)
df.head()

Raw shape: (8688, 26)


,location_id,meal,year,month,day,hour,minute,day_of_week,week_of_year,is_weekend,hour_frac,hour_sin,hour_cos,month_sin,month_cos,is_holiday,holiday_name,special_event,footfall,footfall_raw,waste_kg,waste_kg_target,waste_organic_kg,waste_recyclable_kg,waste_landfill_kg
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,b,closed,2025,1,1,0,0,2,1,0,0.0000,0.0000,1.0000,0.5000,0.8660,1,new year's day,nfl playoff watch party,0,0,0.0000,0.0000,0.0000,0.0000,0.0000
2025-01-01 00:30:00,d,closed,2025,1,1,0,30,2,1,0,0.5000,0.0000,1.0000,0.5000,0.8660,1,new year's day,nfl playoff watch party,4,4,0.0400,0.0400,0.0200,0.0100,0.0100
2025-01-01 01:00:00,d,closed,2025,1,1,1,0,2,1,0,1.0000,0.2588,0.9659,0.5000,0.8660,1,new year's day,nfl playoff watch party,1,1,0.0500,0.0500,0.0200,0.0100,0.0100
2025-01-01 01:30:00,b,closed,2025,1,1,1,30,2,1,0,1.5000,0.2588,0.9659,0.5000,0.8660,1,new year's day,nfl playoff watch party,4,4,0.0500,0.0500,0.0200,0.0100,0.0200
2025-01-01 02:00:00,d,closed,2025,1,1,2,0,2,1,0,2.0000,0.5000,0.8660,0.5000,0.8660,1,new year's day,nfl playoff watch party,0,0,0.0200,0.0200,0.0100,0.0000,0.0100


## Step 2 — Aggregate to Daily Level per Canteen Section

We sum waste and cost variables, take the **mean** foot traffic (because it’s a rate per half‑hour), and preserve flag columns (holiday, special event) – a day is a holiday if any half‑hour is a holiday.

In [196]:
import numpy as np

# Extract date from index, keeping it as a datetime object for easier grouping with meal
df['date'] = df.index.normalize()

# Define aggregation dictionary
agg_dict = {
    'waste_kg': 'sum',                     # total waste per meal time
    'waste_organic_kg': 'sum',
    'waste_recyclable_kg': 'sum',
    'waste_landfill_kg': 'sum',
    'footfall': 'mean',                    # average half-hourly foot traffic per meal time
    'is_holiday': 'max',                   # 1 if any hour within the meal was a holiday
    'special_event': lambda x: ', '.join(x.dropna().unique()) if any(x.notna()) else None  # combine events
}

daily = df.groupby(['date', 'location_id', 'meal']).agg(agg_dict).reset_index()

# Rename columns for clarity
daily.rename(columns={
    'date': 'date',
    'location_id': 'section',
    'meal': 'meal',
    'waste_kg': 'waste_kg',
    'footfall': 'foot_traffic'
}, inplace=True)

print(f"Daily (by meal) shape: {daily.shape}")
display(daily.head())

Daily (by meal) shape: (2618, 10)


,date,section,meal,waste_kg,waste_organic_kg,waste_recyclable_kg,waste_landfill_kg,foot_traffic,is_holiday,special_event
0,2025-01-01,a,closed,0.1800,0.1000,0.0500,0.0400,1.6667,1,nfl playoff watch party
1,2025-01-01,a,dinner,3.1800,1.9900,0.8100,0.3900,12.5000,1,nfl playoff watch party
2,2025-01-01,a,lunch,4.2300,2.2800,1.4300,0.5200,22.6667,1,nfl playoff watch party
3,2025-01-01,b,breakfast,2.9800,1.2200,0.8900,0.8700,6.7500,1,nfl playoff watch party
4,2025-01-01,b,closed,0.2000,0.1000,0.0600,0.0300,1.8333,1,nfl playoff watch party


## Step 3 — Remove Zero‑Waste Days (Optional)

Days with zero waste are rare (canteen closed). We keep them for completeness, but you may drop them if they distort training.

In [197]:
zero_waste_days = daily[daily['waste_kg'] == 0]
print(f"Zero-waste days: {len(zero_waste_days)} ({len(zero_waste_days)/len(daily)*100:.2f}%)")

# We keep them; they represent closed days.
# If needed: daily = daily[daily['waste_kg'] > 0]

Zero-waste days: 1 (0.04%)


## Step 4 — Holiday Flags

`is_holiday` is already present (max aggregated). We also create a binary flag for special events.

In [198]:
# Ensure is_holiday is integer 0/1
daily['is_holiday'] = daily['is_holiday'].astype(int)

# Create a special event flag
daily['has_special_event'] = daily['special_event'].notna().astype(int)

# Drop the combined string column (not needed for modeling)
daily.drop(columns=['special_event'], inplace=True)

daily[['date', 'section', 'waste_kg', 'is_holiday', 'has_special_event']].head()

,date,section,waste_kg,is_holiday,has_special_event
0,2025-01-01,a,0.1800,1,1
1,2025-01-01,a,3.1800,1,1
2,2025-01-01,a,4.2300,1,1
3,2025-01-01,b,2.9800,1,1
4,2025-01-01,b,0.2000,1,1


## Step 5 — Calendar Features

Clean, interpretable features derived from `date`. These are used directly by tree‑based models and as optional regressors for Prophet/SARIMA.

In [199]:
daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.month
daily['day'] = daily['date'].dt.day
daily['day_of_week'] = daily['date'].dt.dayofweek   # Monday=0, Sunday=6
daily['day_of_year'] = daily['date'].dt.dayofyear
daily['week_of_year'] = daily['date'].dt.isocalendar().week
daily['quarter'] = daily['date'].dt.quarter
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)

print("Calendar features added.")
daily[['date', 'year', 'month', 'day_of_week', 'is_weekend']].head()

Calendar features added.


,date,year,month,day_of_week,is_weekend
0,2025-01-01,2025,1,2,0
1,2025-01-01,2025,1,2,0
2,2025-01-01,2025,1,2,0
3,2025-01-01,2025,1,2,0
4,2025-01-01,2025,1,2,0


## Step 6 — Cyclical Encoding (for SARIMA/Prophet optional, but kept for XGBoost)

We add sin/cos transformations of hour‑of‑year and day‑of‑week for models that need cyclicality. For XGBoost, raw integers often work, but these can help.

In [200]:
# Day of week cyclical
daily['dow_sin'] = np.sin(2 * np.pi * daily['day_of_week'] / 7)
daily['dow_cos'] = np.cos(2 * np.pi * daily['day_of_week'] / 7)

# Day of year cyclical (seasonality within year)
daily['doy_sin'] = np.sin(2 * np.pi * daily['day_of_year'] / 365)
daily['doy_cos'] = np.cos(2 * np.pi * daily['day_of_year'] / 365)

daily[['date', 'day_of_week', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos']].head()

,date,day_of_week,dow_sin,dow_cos,doy_sin,doy_cos
0,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
1,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
2,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
3,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
4,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999


## Step 7 — Base Dataset for All Models

We create `waste_features_full.csv` which contains **one row per (date, section)** with all features except lags and rolling stats. This file is suitable for Prophet, SARIMA, and the moving average baseline.

In [201]:
# Select columns for full dataset (no lags)
full_cols = [
    'date', 'section',
    'waste_kg', 'foot_traffic',
    'waste_organic_kg', 'waste_recyclable_kg', 'waste_landfill_kg',
    'is_holiday', 'has_special_event',
    'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend',
    'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos'
]

full_df = daily[full_cols].copy()
full_df.to_csv('data/waste_features_full.csv', index=False)
print(f"Saved waste_features_full.csv with shape {full_df.shape}")

Saved waste_features_full.csv with shape (2618, 21)


## Step 8 — Lag Features (XGBoost / LightGBM only)

Lags give the model direct access to recent history without internal state. We compute lags **within each section** so section A's history never contaminates section B.

| Lag | Reasoning |
|---|---|
| `lag_1` | Yesterday's waste — strongest short‑term signal |
| `lag_7` | Same weekday last week — captures weekly seasonality |
| `lag_14` | Two weeks ago — cross‑checks weekly pattern |
| `lag_28` | Four weeks ago — accounts for monthly rhythms |

In [202]:
# Sort by section and date
daily_sorted = daily.sort_values(['section', 'date']).copy()

# Define lags
lags = [1, 7, 14, 28]
for lag in lags:
    daily_sorted[f'lag_{lag}'] = daily_sorted.groupby('section')['waste_kg'].shift(lag)

print("Lag features added.")
daily_sorted[['section', 'date', 'waste_kg', 'lag_1', 'lag_7', 'lag_14', 'lag_28']].head(10)

Lag features added.


,section,date,waste_kg,lag_1,lag_7,lag_14,lag_28
0,a,2025-01-01,0.1800,NaN,NaN,NaN,NaN
1,a,2025-01-01,3.1800,0.1800,NaN,NaN,NaN
2,a,2025-01-01,4.2300,3.1800,NaN,NaN,NaN
14,a,2025-01-02,0.0600,4.2300,NaN,NaN,NaN
15,a,2025-01-02,1.5000,0.0600,NaN,NaN,NaN
16,a,2025-01-02,1.8200,1.5000,NaN,NaN,NaN
29,a,2025-01-03,1.7100,1.8200,NaN,NaN,NaN
30,a,2025-01-03,0.1200,1.7100,0.1800,NaN,NaN
31,a,2025-01-03,3.3400,0.1200,3.1800,NaN,NaN
32,a,2025-01-03,2.5600,3.3400,4.2300,NaN,NaN


## Step 9 — Rolling Window Statistics (XGBoost / LightGBM only)

Rolling features smooth short‑term noise and encode recent trend/volatility. Windows computed with `min_periods=1` so early rows are not all NaN.

| Feature | What it captures |
|---|---|
| `rolling_mean_7` | Recent 7‑day average — short‑term trend |
| `rolling_mean_14` | 14‑day average — medium‑term trend |
| `rolling_std_7` | 7‑day standard deviation — volatility signal |
| `rolling_max_7` | 7‑day peak — captures spike risk |

In [203]:
# Rolling windows
windows = [7, 14]
for w in windows:
    daily_sorted[f'rolling_mean_{w}'] = daily_sorted.groupby('section')['waste_kg'].transform(
        lambda x: x.rolling(w, min_periods=1).mean()
    )

# Rolling standard deviation (7-day)
daily_sorted['rolling_std_7'] = daily_sorted.groupby('section')['waste_kg'].transform(
    lambda x: x.rolling(7, min_periods=1).std()
)

# Rolling max (7-day)
daily_sorted['rolling_max_7'] = daily_sorted.groupby('section')['waste_kg'].transform(
    lambda x: x.rolling(7, min_periods=1).max()
)

print("Rolling features added.")
daily_sorted[['section', 'date', 'waste_kg', 'rolling_mean_7', 'rolling_std_7', 'rolling_max_7']].head(10)

Rolling features added.


,section,date,waste_kg,rolling_mean_7,rolling_std_7,rolling_max_7
0,a,2025-01-01,0.1800,0.1800,NaN,0.1800
1,a,2025-01-01,3.1800,1.6800,2.1213,3.1800
2,a,2025-01-01,4.2300,2.5300,2.1018,4.2300
14,a,2025-01-02,0.0600,1.9125,2.1143,4.2300
15,a,2025-01-02,1.5000,1.8300,1.8403,4.2300
16,a,2025-01-02,1.8200,1.8283,1.6460,4.2300
29,a,2025-01-03,1.7100,1.8114,1.5033,4.2300
30,a,2025-01-03,0.1200,1.8029,1.5143,4.2300
31,a,2025-01-03,3.3400,1.8257,1.5395,4.2300
32,a,2025-01-03,2.5600,1.5871,1.1959,3.3400


## Step 10 — Location Encoding (XGBoost / LightGBM only)

Tree models need numeric input. We label‑encode `section` (a=0, b=1, c=2, d=3) and also create one‑hot columns for interpretability (optional).

In [204]:
# Label encoding
section_mapping = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
daily_sorted['section_encoded'] = daily_sorted['section'].map(section_mapping)

# One-hot encoding (optional, but good for linear models; XGBoost can handle label encoding)
one_hot = pd.get_dummies(daily_sorted['section'], prefix='section')
daily_sorted = pd.concat([daily_sorted, one_hot], axis=1)

print("Location encoding added.")
daily_sorted[['section', 'section_encoded', 'section_a', 'section_b', 'section_c', 'section_d']].head()

Location encoding added.


,section,section_encoded,section_a,section_b,section_c,section_d
0,a,0,True,False,False,False
1,a,0,True,False,False,False
2,a,0,True,False,False,False
14,a,0,True,False,False,False
15,a,0,True,False,False,False


## Step 11 — Handle NaNs from Lag / Rolling Features

The first `lag_28` rows per section will be NaN (insufficient history). For XGBoost we drop those rows. Prophet/SARIMA use the full dataset without lags, so no issue.

In [205]:
# Check NaN counts
nan_counts = daily_sorted[['lag_1', 'lag_7', 'lag_14', 'lag_28']].isna().sum()
print("NaN counts before dropping:\n", nan_counts)

# Keep only rows where all lags are non‑NaN (i.e., at least 28 days of history per section)
xgb_df = daily_sorted.dropna(subset=['lag_1', 'lag_7', 'lag_14', 'lag_28']).copy()
print(f"XGBoost dataset shape after dropping lag NaNs: {xgb_df.shape}")

# For completeness, also ensure rolling features are non‑NaN (they have min_periods=1, so fine)

NaN counts before dropping:
 lag_1       4
lag_7      28
lag_14     56
lag_28    112
dtype: int64
XGBoost dataset shape after dropping lag NaNs: (2506, 35)


## Step 12 — Final XGBoost / LightGBM Dataset

We select features (excluding date and original section string) and save to CSV. The target is `waste_kg`.

In [206]:
# Define feature columns for XGBoost (exclude non‑feature columns)
exclude_cols = ['date', 'section']  # section is string, we have encoded versions
feature_cols = [col for col in xgb_df.columns if col not in exclude_cols]

# Ensure target is included
if 'waste_kg' not in feature_cols:
    feature_cols = ['waste_kg'] + feature_cols

xgb_export = xgb_df[feature_cols].copy()
xgb_export.to_csv('data/waste_features_xgb.csv', index=False)
print(f"Saved waste_features_xgb.csv with shape {xgb_export.shape}")
print("Features:", list(xgb_export.columns))

Saved waste_features_xgb.csv with shape (2506, 33)
Features: ['meal', 'waste_kg', 'waste_organic_kg', 'waste_recyclable_kg', 'waste_landfill_kg', 'foot_traffic', 'is_holiday', 'has_special_event', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'rolling_max_7', 'section_encoded', 'section_a', 'section_b', 'section_c', 'section_d']


## Step 13 — Per‑Section CSVs (for SARIMA univariate series)

SARIMA works on a single time series. We save one CSV per section containing `date` and `waste_kg` (plus optional exog variables).

In [207]:
sections = daily['section'].unique()
for sec in sections:
    sec_df = daily[daily['section'] == sec][['date', 'waste_kg', 'foot_traffic', 'is_holiday', 'has_special_event']].copy()
    sec_df.sort_values('date', inplace=True)
    sec_df.to_csv(f'data/per_section/waste_section_{sec}.csv', index=False)
    print(f"Saved section {sec} with {len(sec_df)} rows")

# Also save a total waste series (sum across sections) for SARIMA on total
total_waste = daily.groupby('date')['waste_kg'].sum().reset_index()
total_waste.to_csv('data/per_section/waste_total.csv', index=False)
print("Saved total waste series (waste_total.csv)")

Saved section a with 656 rows
Saved section b with 657 rows
Saved section c with 658 rows
Saved section d with 647 rows
Saved total waste series (waste_total.csv)


## Preprocessing Complete

Three datasets have been exported:
1. `data/waste_features_full.csv` – for Prophet, SARIMA, MA
2. `data/waste_features_xgb.csv` – for XGBoost / LightGBM (with lags, rollings, encodings)
3. `data/per_section/` – separate CSVs for each section (including total) for SARIMA
